In [2]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), 'C\u1ea7n GPU!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb >= 15, f'C\u00e2n \u226515GB VRAM (hi\u1ec7n c\u00f3 {vram_gb:.1f}GB)'

# Check SM version for Flash Attention 2 support
# FA2 requires Ampere+ (SM >= 8.0): A100, A10, 3090, 4090
# T4 (Turing, SM 7.5) does NOT support FA2 - will use SDPA instead
major, minor = torch.cuda.get_device_capability()
HAS_FA2 = major >= 8
print(f'SM version: {major}.{minor}')
print(f'Flash Attention 2: {"supported" if HAS_FA2 else "NOT supported (will use SDPA)"}')


GPU: Tesla T4
VRAM: 15.6 GB
SM version: 7.5
Flash Attention 2: NOT supported (will use SDPA)


In [3]:
# Unsloth: kéo sẵn torch, transformers, trl, peft, bitsandbytes, accelerate,
#           flash-attn (wheel prebuilt) với version tương thích
!pip install -q unsloth

# Deps còn lại cho data/eval (không bị Unsloth bundle)
!pip install -q datasets sentencepiece scikit-learn pandas python-dotenv pyyaml huggingface-hub protobuf tqdm

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
HF_REPO = user_secrets.get_secret("HF_REPO")


In [5]:
import os
import re
import json
import glob
import time
import math
from collections import Counter, defaultdict
def load_json_files(dataset_path):
    files = glob.glob(os.path.join(dataset_path, "*.json"))
    print(f"Found {len(files)} JSON files in '{dataset_path}'")
    items = []
    for p in files:
        try:
            with open(p, "r", encoding="utf-8") as f:
                items.append(json.load(f))
        except Exception as e:
            print(f"Error reading {p}: {e}")
    return items
text1= load_json_files("/kaggle/input/tempo-run-2025-run-with-ai-break-limits/Dataset/Dataset/train")
text1[0]


Found 1500 JSON files in '/kaggle/input/tempo-run-2025-run-with-ai-break-limits/Dataset/Dataset/train'


{'url': 'https://thanhnien.vn/sap-khai-mac-trien-lam-nganh-giay-bao-bi-dien-nang-luong-tu-dong-hoa-185240426102412567.htm',
 'title': 'Sắp khai mạc triển lãm ngành giấy, bao bì, điện, năng lượng, tự động hóa',
 'content': 'Từ ngày 8 - 10.5 tại Trung tâm Triển lãm quốc tế WTC Expo tỉnh Bình Dương sẽ diễn ra Triển lãm quốc tế giấy và bao bì Việt Nam - VPPE 2024 và Triển lãm quốc tế ngành điện, năng lượng, máy móc thiết bị công nghiệp, tự động hóa Việt Nam lần 3 - EMA Vietnam 2024.\nNgành giấy là một trong những ngành kinh tế lớn, có tầm quan trọng trong cơ cấu kinh tế nói chung và kinh tế công nghiệp nói riêng. Bên cạnh là ngành phụ trợ quan trọng cho các ngành khác như điện tử, may mặc, da giày, thực phẩm, đồ gỗ…, thì ngành giấy còn cung cấp nhiều sản phẩm cho mục đích đa dạng từ hàng hóa tiêu dùng thiết yếu đến các hoạt động văn hóa, xã hội, giáo dục, y tế, thông tin truyền thông… Trong định hướng phát triển, đến năm 2030, ngành giấy Việt Nam sẽ trở thành ngành sản xuất lớn của khu vực

In [7]:
# 7. Build SFT JSONL — inline tất cả logic
import json, glob, random
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

REPO_DIR = Path('/kaggle/working/')
TRAIN_DIR = Path('/kaggle/input/tempo-run-2025-run-with-ai-break-limits/Dataset/Dataset/train')
OUT_DIR = REPO_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = 'Bạn là hệ thống trả lời trắc nghiệm. Chỉ xuất duy nhất 1 ký tự A/B/C/D.'

def format_choices(choices):
    """Render choices deterministically A-D."""
    ordered = []
    for k in ['A', 'B', 'C', 'D']:
        if k in choices and choices[k] is not None:
            ordered.append(f'{k}: {choices[k]}')
    return '\n'.join(ordered)

def build_user_instruction(title, content, question, choices):
    """Build user-message text cho 1 MCQ item."""
    ct = format_choices(choices)
    return (
        'Bạn là hệ thống trả lời trắc nghiệm. Hãy đọc văn bản và câu hỏi, '
        'chỉ chọn **một đáp án duy nhất** từ A/B/C/D, không giải thích, không thêm nội dung khác.\n\n'
        'Văn bản:\n'
        f'Tiêu đề: {title}\n\n'
        f'Nội dung: {content}\n\n'
        'Câu hỏi:\n'
        f'{question}\n\n'
        'Các lựa chọn:\n'
        f'{ct}\n\n'
        'Chỉ trả lời đúng 1 ký tự: A, B, C hoặc D.'
    )

def build_messages(system_prompt, user_instruction, assistant=None):
    msgs = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_instruction},
    ]
    if assistant is not None:
        msgs.append({'role': 'assistant', 'content': assistant})
    return msgs

# Load all JSON files and build rows
rows = []
dropped = 0
for path in sorted(glob.glob(str(TRAIN_DIR / '*.json'))):
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    content = (d.get('content:') or d.get('content') or '').strip()
    title = (d.get('title:') or d.get('title') or '').strip()
    questions = d.get('questions') or []
    if not content or not questions:
        dropped += 1
        continue
    for idx, q in enumerate(questions, start=1):
        question = (q.get('question') or '').strip()
        choices = q.get('choices') or {}
        raw_label = q.get('correct_answer')
        label = raw_label.strip().upper()[:1] if raw_label else None
        if not question or not choices:
            continue
        if label is not None and label not in {'A', 'B', 'C', 'D'}:
            continue
        user = build_user_instruction(title, content, question, choices)
        assistant = label if label is not None else None
        msgs = build_messages(SYSTEM_PROMPT, user, assistant=assistant)
        row = {
            'messages': msgs,
            'title': title, 'content': content,
            'question': question, 'choices': choices,
        }
        if label is not None:
            row['label'] = label
        rows.append(row)

print(f'Built {len(rows)} rows (dropped {dropped} docs)')

# Stratified split 90/10
labels = [r.get('label', 'UNK') for r in rows]
train_rows, eval_rows = train_test_split(
    rows, test_size=0.1, random_state=3407, shuffle=True, stratify=labels,
)
print(f'Train: {len(train_rows)}  Eval: {len(eval_rows)}')
print(f'Train label dist: {dict(Counter(r.get("label","UNK") for r in train_rows))}')
print(f'Eval  label dist: {dict(Counter(r.get("label","UNK") for r in eval_rows))}')

def write_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

write_jsonl(train_rows, OUT_DIR / 'train.jsonl')
write_jsonl(eval_rows, OUT_DIR / 'eval.jsonl')
print(f'Wrote {OUT_DIR / "train.jsonl"} and {OUT_DIR / "eval.jsonl"}')


Built 4491 rows (dropped 3 docs)
Train: 4041  Eval: 450
Train label dist: {'B': 1638, 'A': 1257, 'D': 144, 'C': 1002}
Eval  label dist: {'C': 112, 'B': 182, 'A': 140, 'D': 16}
Wrote /kaggle/working/data/processed/train.jsonl and /kaggle/working/data/processed/eval.jsonl


In [8]:
# 8. Balance labels — inline balance_via_reorder + merge
# Reuse: build_user_instruction, build_messages, write_jsonl, read_jsonl,
#         train_test_split from cell 7
from collections import Counter
import random

PROC_DIR = REPO_DIR / 'data' / 'processed'
FINAL_DIR = PROC_DIR / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

def reorder_row_for_label(row, target_label):
    """Rotate choices sao cho correct answer lands on target_label."""
    orig_label = row.get('label', '')
    if orig_label not in {'A','B','C','D'} or target_label not in {'A','B','C','D'}:
        return row
    choices = row.get('choices') or {}
    if not all(k in choices for k in ('A','B','C','D')):
        return row
    shift = (ord(target_label) - ord(orig_label)) % 4
    if shift == 0:
        return row
    new_choices = {}
    for k, v in choices.items():
        if k in {'A','B','C','D'}:
            new_choices[chr(ord('A') + (ord(k) - ord('A') + shift) % 4)] = v
        else:
            new_choices[k] = v
    new_user = build_user_instruction(
        row.get('title',''), row.get('content',''),
        row.get('question',''), new_choices,
    )
    new_msgs = build_messages(SYSTEM_PROMPT, new_user, assistant=target_label)
    new_row = dict(row)
    new_row['messages'] = new_msgs
    new_row['choices'] = new_choices
    new_row['label'] = target_label
    return new_row

def balance_via_reorder(rows, seed=3407):
    """Rebalance by rotating choices so label cycles A->B->C->D."""
    rng = random.Random(seed)
    indices = list(range(len(rows)))
    rng.shuffle(indices)
    target_labels = ['A', 'B', 'C', 'D']
    return [reorder_row_for_label(rows[i], target_labels[i % 4]) for i in indices]

# Load train.jsonl, balance, merge with eval, re-split
rows = read_jsonl(PROC_DIR / 'train.jsonl')
pre_dist = dict(Counter(r.get('label','UNK') for r in rows))
print(f'Before balance: {pre_dist}')

balanced = balance_via_reorder(rows, seed=3407)
post_dist = dict(Counter(r.get('label','UNK') for r in balanced))
print(f'After balance:  {post_dist}')

eval_rows = read_jsonl(PROC_DIR / 'eval.jsonl')
all_rows = balanced + eval_rows
print(f'Total: {len(all_rows)} (balanced {len(balanced)} + eval {len(eval_rows)})')

labels = [r.get('label','UNK') for r in all_rows]
final_train, final_eval = train_test_split(
    all_rows, test_size=0.1, random_state=3407, shuffle=True, stratify=labels,
)
print(f'Final train: {len(final_train)}  eval: {len(final_eval)}')

write_jsonl(final_train, FINAL_DIR / 'train.jsonl')
write_jsonl(final_eval, FINAL_DIR / 'eval.jsonl')
print(f'Wrote {FINAL_DIR}')


Before balance: {'B': 1638, 'A': 1257, 'D': 144, 'C': 1002}
After balance:  {'A': 1011, 'D': 1010, 'C': 1010, 'B': 1010}
Total: 4491 (balanced 4041 + eval 450)
Final train: 4041  eval: 450
Wrote /kaggle/working/data/processed/final


In [ ]:
# 9. Load Unsloth model + LoRA
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen3-0.6B'
MAX_SEQ_LENGTH = 2048
LORA_R = 32
LORA_ALPHA = 16
LORA_DROPOUT = 0
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# Load model + tokenizer — Unsloth tự xử lý FA2 + Triton kernels
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto: bf16 trên A100, fp16 trên cũ hơn
    load_in_4bit=True,
)
print(f'Loaded {MODEL_NAME}, vocab={tokenizer.vocab_size}')

# Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)
model.print_trainable_parameters()


In [ ]:
# 10. Render chat text — demo fix thinking mode
# Reuse: train_rows from cell 7

sample = final_train[0]  # từ cell 8
msgs = sample['messages']
print('=== Messages ===')
for m in msgs:
    print(f'  [{m["role"]}] {m["content"][:80]}...')

prefix_msgs = [m for m in msgs if m['role'] != 'assistant']

# Cách CŨ (không fix) — model muốn sinh <think>
old_render = tokenizer.apply_chat_template(
    prefix_msgs, tokenize=False, add_generation_prompt=True,
)
print('\n=== OLD (enable_thinking mặc định = True) ===')
print(repr(old_render[-80:]))

# Cách MỚI (fix) — enable_thinking=False
new_render = tokenizer.apply_chat_template(
    prefix_msgs, tokenize=False, add_generation_prompt=True,
    enable_thinking=False,
)
print('\n=== NEW (enable_thinking=False) ===')
print(repr(new_render[-80:]))

# Render cho training: prefix + assistant content + EOS
def render_chat_for_training(tokenizer, messages, enable_thinking=False):
    """Render messages thành 1 string cho SFT.
    Tách assistant message, render prefix với add_generation_prompt=True,
    rồi append assistant_content + eos.
    """
    assistant_content = None
    prefix = []
    for m in messages:
        if m['role'] == 'assistant':
            assistant_content = m['content']
        else:
            prefix.append(m)
    text = tokenizer.apply_chat_template(
        prefix, tokenize=False, add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    if assistant_content is not None:
        text += assistant_content
    eos = tokenizer.eos_token
    if eos and not text.endswith(eos):
        text += eos
    return text

train_text = render_chat_for_training(tokenizer, msgs, enable_thinking=False)
print('\n=== Training text (enable_thinking=False) ===')
print(repr(train_text[-120:]))


In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
# Reuse: render_chat_for_training, read_jsonl from cell 10 / cell 7

TRAIN_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'train.jsonl'
EVAL_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'eval.jsonl'
OUTPUT_DIR = REPO_DIR / 'artifacts' / 'unsloth_qwen3_0_6b'

train_rows = read_jsonl(TRAIN_JSONL)
eval_rows = read_jsonl(EVAL_JSONL)
print(f'Train: {len(train_rows)}  Eval: {len(eval_rows)}')

# Render to text with enable_thinking=False
def to_text(row):
    return {'text': render_chat_for_training(tokenizer, row['messages'], enable_thinking=False)}

train_ds = Dataset.from_list(train_rows).map(
    to_text, remove_columns=list(train_rows[0].keys()),
)
eval_ds = Dataset.from_list(eval_rows).map(
    to_text, remove_columns=list(eval_rows[0].keys()),
)
print(f'Dataset: train={len(train_ds)} eval={len(eval_ds)}')
print(f'Sample text (last 100 chars): {repr(train_ds[0]["text"][-100:])}')


# --- BẮT ĐẦU PHẦN FIX ---

# 1. Khai báo Data Collator để chỉ tính loss phần assistant
# response_template = "<|im_start|>assistant\n"
# collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# 2. SFT Config chuẩn
sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # effective batch = 32
    num_train_epochs=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type='cosine',
    logging_steps=20,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    max_grad_norm=0.3,
    optim='adamw_8bit',
    
    # FIX: Tự động chạy fp16 trên T4, bf16 trên A100/H100
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    gradient_checkpointing=True,
    dataset_text_field='text',
    packing=False,
    max_seq_length=MAX_SEQ_LENGTH, # FIX: trl bản mới dùng max_seq_length
    eval_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=[],
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

# Kích hoạt tính năng chỉ tính loss trên câu trả lời (completion_only_loss)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",    # Dấu hiệu nhận biết câu hỏi
    response_part="<|im_start|>assistant\n", # Dấu hiệu nhận biết câu trả lời
)


# --- KẾT THÚC PHẦN FIX ---


print('Starting training...')
trainer.train()

# Save LoRA adapter
adapter_dir = OUTPUT_DIR / 'adapter'
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f'Saved adapter to {adapter_dir}')

# Save merged 16-bit (optional, cho standalone inference)
merged_dir = OUTPUT_DIR / 'merged_16bit'
merged_dir.mkdir(parents=True, exist_ok=True)
try:
    model.save_pretrained_merged(str(merged_dir), tokenizer, save_method='merged_16bit')
    print(f'Saved merged 16-bit to {merged_dir}')
except Exception as e:
    print(f'save_pretrained_merged failed: {e}')

In [24]:
import gc
import torch

# 1. Xóa biến trainer nếu nó vẫn còn tồn tại trong RAM
if 'trainer' in globals():
    del trainer

# 2. Xóa sạch bộ nhớ đệm của GPU
torch.cuda.empty_cache()

# 3. Ép Python dọn rác hệ thống
gc.collect()

print("Đã dọn dẹp VRAM! Sẵn sàng chạy Eval.")

Đã dọn dẹp VRAM! Sẵn sàng chạy Eval.


In [18]:
import torch
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
from unsloth import FastLanguageModel # Dùng Unsloth thay vì AutoModel

# Tái sử dụng các đường dẫn cũ
# ADAPTER_PATH = REPO_DIR / 'artifacts' / 'unsloth_qwen3_0_6b' / 'adapter'
ADAPTER_PATH = Path('/kaggle/input/datasets/phcthnho/data-model-tempo/artifacts/unsloth_qwen3_0_6b/adapter')
EVAL_JSONL = REPO_DIR / 'data' / 'processed' / 'final' / 'eval.jsonl'
MAX_LENGTH = 2048
BATCH_SIZE = 4
SYSTEM_PROMPT = 'Bạn là hệ thống trả lời trắc nghiệm. Chỉ xuất duy nhất 1 ký tự A/B/C/D.'

print('Loading model directly via Unsloth...')
# 1. Unsloth cực kỳ thông minh: bạn chỉ cần truyền ADAPTER_PATH, 
# nó sẽ tự đọc config để biết phải kéo base model Qwen nào về.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_PATH),
    max_seq_length=MAX_LENGTH,
    dtype=None,          # Tự động chọn fp16 cho Tesla T4
    load_in_4bit=True,   # Load ở 4-bit (không cần merge)
)

# 2. BẬT TÍNH NĂNG TĂNG TỐC INFERENCE CỦA UNSLOTH (Quan trọng!)
FastLanguageModel.for_inference(model) 

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Fix an toàn: Chuyển padding sang bên trái để logits luôn nằm ở token cuối cùng bên phải
tokenizer.padding_side = "left" 

# --- Phần Logic Lấy Token IDs (Giữ nguyên của bạn) ---
letter_ids = {}
for ch in ['A', 'B', 'C', 'D']:
    ids = tokenizer(ch, add_special_tokens=False).input_ids
    assert len(ids) == 1, f"Letter '{ch}' is not single token (got {ids})"
    letter_ids[ch] = ids[0]
print(f'Letter token IDs: {letter_ids}')

def to_chat_prompt(tokenizer, instruction, system_prompt):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': instruction},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )

eval_rows = read_jsonl(EVAL_JSONL)
print(f'Eval rows: {len(eval_rows)}')

prompts = []
labels = []
for r in eval_rows:
    instr = r['messages'][1]['content']  
    prompts.append(to_chat_prompt(tokenizer, instr, SYSTEM_PROMPT))
    labels.append(r['label'])

# --- Phần Đánh Giá (Cấu trúc giữ nguyên, nhưng sẽ chạy nhanh hơn rất nhiều) ---
correct = 0
preds = []
with torch.no_grad():
    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc='Evaluating'):
        ps = prompts[i:i+BATCH_SIZE]
        ls = labels[i:i+BATCH_SIZE]
        
        enc = tokenizer(ps, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(model.device)
        
        # --- THE FIX ---
        # Chỉ yêu cầu model tính toán logit cho token cuối cùng.
        # Điều này tiết kiệm hàng GB VRAM!
        outputs = model(**enc, num_logits_to_keep=1)
        
        # Vì chúng ta chỉ keep 1 logit, chiều sequence của tensor lúc này là 1.
        # Do đó, ta lấy index 0 thay vì -1.
        next_logits = outputs.logits[:, 0, :] 
        # ---------------
        
        cand_ids = torch.tensor(list(letter_ids.values()), device=outputs.logits.device)
        cand_logits = next_logits[:, cand_ids]
        
        pred_idx = cand_logits.argmax(dim=1)
        idx2letter = list(letter_ids.keys())
        batch_preds = [idx2letter[i.item()] for i in pred_idx]
        preds.extend(batch_preds)
        
        for p, g in zip(batch_preds, ls):
            correct += int(p == g)

# In kết quả
acc = 100.0 * correct / len(eval_rows)
print(f'\n=== EVAL RESULTS ===')
print(f'Total: {len(eval_rows)}')
print(f'Correct: {correct}')
print(f'Accuracy: {acc:.2f}%')

# Confusion matrix
cm = defaultdict(lambda: Counter())
for p, g in zip(preds, labels):
    cm[g][p] += 1
letters_sorted = sorted(set(cm.keys()) | set(preds))
print(f'\nPred dist: {dict(Counter(preds))}')
print('Confusion (rows=label, cols=pred):')
header = '     ' + '  '.join(f'{c:>5}' for c in letters_sorted)
print(header)
for lab in letters_sorted:
    if lab not in cm:
        continue
    row = [cm[lab][p] for p in letters_sorted]
    print(f'{lab:>3}: ' + '  '.join(f'{n:>5}' for n in row))

Loading model directly via Unsloth...
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Letter token IDs: {'A': 32, 'B': 33, 'C': 34, 'D': 35}
Eval rows: 450


Evaluating:   2%|▏         | 2/113 [00:02<01:59,  1.08s/it]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and wil


=== EVAL RESULTS ===
Total: 450
Correct: 374
Accuracy: 83.11%

Pred dist: {'C': 118, 'A': 118, 'D': 97, 'B': 117}
Confusion (rows=label, cols=pred):
         A      B      C      D
  A:    96      5      9      5
  B:     9    100      4      7
  C:     9      4     96      3
  D:     4      8      9     82


In [19]:
#!/bin/bash
!kaggle competitions leaderboard tempo-run-2025-run-with-ai-break-limits -d

100%|██████████████████████████████████████| 1.53k/1.53k [00:00<00:00, 4.90MB/s]



In [25]:
# 13. Generate submission
import json, glob
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Reuse: to_chat_prompt, letter_ids, model, tokenizer, read_jsonl from cell 12
#         build_user_instruction from cell 7

TEST_DIR = Path('/kaggle/input/datasets/phcthnho/tempo-2025/Dataset/Dataset/test')
OUT_CSV = REPO_DIR / 'submissions' / 'sub_unsloth.csv'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
MAX_LENGTH = 2048
BATCH_SIZE = 4
SYSTEM_PROMPT = 'Bạn là hệ thống trả lời trắc nghiệm. Chỉ xuất duy nhất 1 ký tự A/B/C/D.'

# Walk test JSON files — same logic as infer.py
items = []
for path in sorted(glob.glob(str(TEST_DIR / '*.json'))):
    article_id = Path(path).stem
    with open(path, encoding='utf-8') as f:
        d = json.load(f)
    content = (d.get('content:') or d.get('content') or '').strip()
    title = (d.get('title:') or d.get('title') or '').strip()
    questions = d.get('questions') or []
    if not content or not questions:
        continue
    for idx, q in enumerate(questions, start=1):
        question = (q.get('question') or '').strip()
        choices = q.get('choices') or {}
        if not question or not choices:
            continue
        items.append({
            'article_id': article_id,
            'qid': idx,
            'row_id': f'{article_id}__q{idx}',
            'title': title,
            'content': content,
            'question': question,
            'choices': choices,
        })

print(f'Test items: {len(items)}')

# Build prompts
prompts = []
for it in items:
    instr = build_user_instruction(it['title'], it['content'], it['question'], it['choices'])
    prompts.append(to_chat_prompt(tokenizer, instr, SYSTEM_PROMPT))

# Predict — same logits mode as eval
results = []
with torch.no_grad():
    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc='Inferring'):
        ps = prompts[i:i+BATCH_SIZE]
        enc = tokenizer(ps, return_tensors='pt', padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(model.device)
        
        # 1. Bring back the memory optimization to prevent OOM
        outputs = model(**enc, num_logits_to_keep=1)
        
        # 2. Because we only kept 1 logit, the sequence dimension is 1.
        # We don't need any complex attention_mask math. Just grab index 0!
        next_logits = outputs.logits[:, 0, :]
        
        cand_ids = torch.tensor(list(letter_ids.values()), device=outputs.logits.device)
        cand_logits = next_logits[:, cand_ids]
        
        pred_idx = cand_logits.argmax(dim=1)
        idx2letter = list(letter_ids.keys())
        
        for j, p in enumerate(pred_idx):
            results.append({
                'row_id': items[i + j]['row_id'],
                'answer': idx2letter[p.item()],
            })
        del enc, outputs, next_logits, cand_logits
        torch.cuda.empty_cache()

df = pd.DataFrame(results)
df = df[['row_id', 'answer']]
df.to_csv(OUT_CSV, index=False)
print(f'\nWrote {len(df)} rows to {OUT_CSV}')
print(f'Pred dist: {df["answer"].value_counts().to_dict()}')
print(f'Duplicates: {df["row_id"].duplicated().sum()}')
print(df.head(10))


Test items: 1488


Inferring: 100%|██████████| 372/372 [04:54<00:00,  1.27it/s]


Wrote 1488 rows to /kaggle/working/submissions/sub_unsloth.csv
Pred dist: {'B': 526, 'A': 459, 'C': 429, 'D': 74}
Duplicates: 0
         row_id answer
0  0164aa98__q1      C
1  0164aa98__q2      C
2  0164aa98__q3      C
3  019aeed2__q1      C
4  019aeed2__q2      B
5  019aeed2__q3      B
6  021140a6__q1      B
7  021140a6__q2      B
8  021140a6__q3      B
9  0220be76__q1      C


In [26]:
import pandas as pd
import zipfile
from sklearn.metrics import accuracy_score, confusion_matrix

# Đường dẫn tới file dự đoán của bạn và file ZIP chứa đáp án
PRED_CSV_PATH = OUT_CSV # Lấy biến OUT_CSV từ đoạn code trên của bạn
TRUTH_ZIP_PATH = '/kaggle/working/tempo-run-2025-run-with-ai-break-limits.zip'

# 1. Đọc file kết quả dự đoán của model
df_pred = pd.read_csv(PRED_CSV_PATH)
print(f"📊 Đã tải {len(df_pred)} dòng từ file dự đoán.")

# 2. Đọc trực tiếp file đáp án từ trong file ZIP
with zipfile.ZipFile(TRUTH_ZIP_PATH, 'r') as z:
    file_names = z.namelist()
    print(f"📁 Các file có trong ZIP: {file_names}")
    
    # Tìm file CSV chứa đáp án (thường là solution.csv, test_labels.csv hoặc test.csv)
    # Lấy file csv đầu tiên tìm thấy, hoặc bạn có thể gõ trực tiếp tên file vào đây
    csv_files = [f for f in file_names if f.endswith('.csv')]
    
    if not csv_files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong ZIP!")
        
    truth_file_name = csv_files[0] 
    print(f"📄 Đang đọc đáp án gốc từ: {truth_file_name}")
    
    with z.open(truth_file_name) as f:
        df_true = pd.read_csv(f)

# 3. Tiền xử lý để ghép 2 bảng (Merge)
# LƯU Ý: Đổi tên 'row_id' và 'answer' bên dưới nếu file đáp án gốc dùng tên cột khác (ví dụ: 'Id' và 'Expected')
TRUTH_ID_COL = 'row_id' 
TRUTH_ANS_COL = 'answer'

if TRUTH_ID_COL not in df_true.columns or TRUTH_ANS_COL not in df_true.columns:
    print(f"⚠️ Cảnh báo: File gốc có các cột {df_true.columns.tolist()}.")
    print(f"Hãy sửa TRUTH_ID_COL và TRUTH_ANS_COL trong code cho khớp với file gốc nhé!")
else:
    # Ghép 2 bảng lại dựa trên ID của câu hỏi
    df_merged = pd.merge(df_pred, df_true, left_on='row_id', right_on=TRUTH_ID_COL, suffixes=('_pred', '_true'))
    
    # 4. Tính toán và in kết quả
    y_pred = df_merged['answer_pred'].astype(str).str.upper()
    y_true = df_merged[TRUTH_ANS_COL].astype(str).str.upper()
    
    acc = accuracy_score(y_true, y_pred) * 100
    
    print("\n" + "="*30)
    print("🏆 KẾT QUẢ TRÊN TẬP TEST KAGGLE 🏆")
    print("="*30)
    print(f"Tổng số câu hỏi: {len(df_merged)}")
    print(f"Số câu đúng:     {sum(y_pred == y_true)}")
    print(f"Accuracy:        {acc:.2f}%")
    
    # In Confusion Matrix cho đẹp
    labels = sorted(list(set(y_true) | set(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=[f"True_{l}" for l in labels], columns=[f"Pred_{l}" for l in labels])
    
    print("\n🔍 Confusion Matrix:")
    print(cm_df)
    
    # Xuất ra các câu làm sai để phân tích thêm
    df_wrong = df_merged[y_pred != y_true]
    print(f"\n❌ Model làm sai {len(df_wrong)} câu. Xem thử 5 câu đầu tiên:")
    print(df_wrong[['row_id', 'answer_true', 'answer_pred']].head(5))

📊 Đã tải 1488 dòng từ file dự đoán.
📁 Các file có trong ZIP: ['tempo-run-2025-run-with-ai-break-limits-publicleaderboard-2026-07-07T01:08:15.csv']
📄 Đang đọc đáp án gốc từ: tempo-run-2025-run-with-ai-break-limits-publicleaderboard-2026-07-07T01:08:15.csv
⚠️ Cảnh báo: File gốc có các cột ['Rank', 'TeamId', 'TeamName', 'LastSubmissionDate', 'Score', 'SubmissionCount', 'TeamMemberUserNames'].
Hãy sửa TRUTH_ID_COL và TRUTH_ANS_COL trong code cho khớp với file gốc nhé!
